# **Analyse du drift & de la performance**

Ce notebook a pour objectif :

- **d’évaluer le data drift** entre :
  - les données de référence (jeu d’entraînement ou échantillon représentatif)
  - les données de production issues des logs de l’API

  Le data drift permet de détecter si les distributions des variables évoluent dans le temps.  
  Ces changements peuvent dégrader les performances du modèle, notamment lorsque les distributions de production s’éloignent de celles observées lors de l’entraînement.

  Pour cela, nous utilisons la librairie **Evidently AI**, qui permet d’analyser le drift **colonne par colonne**, après avoir soigneusement harmonisé les colonnes entre les deux jeux de données (types, valeurs manquantes, colonnes constantes, etc.).

- **d’évaluer la performance du pipeline de prédiction**, non pas uniquement au niveau du modèle, mais surtout au niveau du **preprocessing**, afin d’identifier les véritables axes d’optimisation.

  En effet, dans un contexte MLOps, la latence d’un modèle en production dépend autant du modèle que du pipeline de transformation.  
  Nous mesurons donc :
  - le temps de preprocessing,
  - le temps de prédiction du modèle,
  - la différence entre appels unitaires et prédiction en batch,
  - et nous analysons les résultats du profiling pour identifier les goulots d’étranglement.



In [60]:
import pandas as pd
import cProfile
import pstats
import joblib
import time

from evidently.report import Report
from evidently.metrics import ColumnDriftMetric

## **Analyse du Data Drift avec Evidently**

In [61]:
# Chargement du pipeline
pipe = joblib.load("../BestModel/pipeline_complet.joblib")

# Données de référence (version simplifiée)
df_reference_raw = joblib.load("../data/app_test_small.joblib")

# Données de production (logs unitaires)
df_production_raw = pd.read_parquet("../logs/predictions_log.parquet")

# Données de production (logs batch)
df_batch_raw = pd.read_parquet("../logs/predictions_log.parquet")

df_reference_raw.shape, df_production_raw.shape, df_batch_raw.shape

((500, 327), (213, 406), (213, 406))

### Sélection des colonnes communes entre référence production

In [62]:
# Transformer les colonnes de référence
df_reference_transformed = pipe[:-1].transform(df_reference_raw)
df_reference_transformed = pd.DataFrame(df_reference_transformed, columns=pipe[:-1].get_feature_names_out())
df_reference_transformed.columns


Index(['num__SK_ID_CURR', 'num__BUREAU_SK_ID_BUREAU_max',
       'num__POS_SK_DPD_min', 'num__CC_SK_DPD_DEF_max',
       'num__PREV_NAME_TYPE_SUITE_Family', 'num__PREV_DAYS_FIRST_DRAWING_sum',
       'num__PREV_NAME_GOODS_CATEGORY_Other', 'num__DAYS_LAST_PHONE_CHANGE',
       'num__CC_CNT_DRAWINGS_OTHER_CURRENT_mean',
       'num__PREV_DAYS_FIRST_DRAWING_max',
       ...
       'bool__FLAG_DOCUMENT_12', 'bool__FLAG_DOCUMENT_13',
       'bool__FLAG_DOCUMENT_14', 'bool__FLAG_DOCUMENT_15',
       'bool__FLAG_DOCUMENT_16', 'bool__FLAG_DOCUMENT_17',
       'bool__FLAG_DOCUMENT_18', 'bool__FLAG_DOCUMENT_19',
       'bool__FLAG_DOCUMENT_20', 'bool__FLAG_DOCUMENT_21'],
      dtype='object', length=382)

In [63]:
df_production_raw.columns

Index(['num__SK_ID_CURR', 'num__BUREAU_SK_ID_BUREAU_max',
       'num__POS_SK_DPD_min', 'num__CC_SK_DPD_DEF_max',
       'num__PREV_NAME_TYPE_SUITE_Family', 'num__PREV_DAYS_FIRST_DRAWING_sum',
       'num__PREV_NAME_GOODS_CATEGORY_Other', 'num__DAYS_LAST_PHONE_CHANGE',
       'num__CC_CNT_DRAWINGS_OTHER_CURRENT_mean',
       'num__PREV_DAYS_FIRST_DRAWING_max',
       ...
       'ram_percent', 'system_load', 'num_threads', 'method', 'status',
       'error_message', 'latency_ms', 'request_size_bytes',
       'response_size_bytes', 'batch_size_y'],
      dtype='object', length=406)

In [64]:
common_cols = df_reference_transformed.columns.intersection(df_production_raw.columns)
len(common_cols)

382

In [65]:
df_reference = df_reference_transformed[common_cols].copy()
df_production = df_production_raw[common_cols].copy()

df_reference.shape, df_production.shape

((500, 382), (213, 382))

### Harmonisation des colonnes

In [66]:
for col in common_cols:
    # bool → int
    if df_reference[col].dtype == bool:
        df_reference[col] = df_reference[col].astype(int)

    if df_production[col].dtype == bool:
        df_production[col] = df_production[col].astype(int)

    ref_num = pd.api.types.is_numeric_dtype(df_reference[col])
    prod_num = pd.api.types.is_numeric_dtype(df_production[col])

    # Les deux numériques
    if ref_num and prod_num:
        df_reference[col] = pd.to_numeric(df_reference[col], errors="coerce").astype("float64")
        df_production[col] = pd.to_numeric(df_production[col], errors="coerce").astype("float64")

    # Sinon → string
    else:
        df_reference[col] = df_reference[col].fillna("MISSING").astype(str)
        df_production[col] = df_production[col].fillna("MISSING").astype(str)


### Suppression des colonnes vides et constantes

In [67]:
valid_cols = [
    col for col in common_cols
    if df_reference[col].notna().sum() > 0 and df_production[col].notna().sum() > 0]

non_constant_cols = [
    col for col in valid_cols
    if df_reference[col].nunique(dropna=True) > 1
    and df_production[col].nunique(dropna=True) > 1]

df_reference = df_reference[non_constant_cols]
df_production = df_production[non_constant_cols]

print(len(non_constant_cols))
print(len(df_production))
print(len(df_reference))


275
213
500


### Drift par colonne

In [68]:
drift_results = []

for col in non_constant_cols:
    report = Report(metrics = [ColumnDriftMetric(column_name = col)])

    try:
        report.run(reference_data = df_reference[[col]], current_data = df_production[[col]])

        result = report.as_dict()
        drift_detected = result["metrics"][0]["result"]["drift_detected"]

        drift_results.append({
            "colonne": col,
            "drift_detected": drift_detected})

    except Exception as e:
        drift_results.append({"colonne": col, "drift_detected": "ERREUR", "exception": str(e)})

df_drift = pd.DataFrame(drift_results)
print(df_drift.shape)
report_drift = report


c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: Runt

(275, 2)


### Résumé

In [69]:
from evidently.metrics import DatasetDriftMetric
report = Report(metrics=[DatasetDriftMetric()])
report.run(reference_data=df_reference, current_data=df_production)

res = report.as_dict()

c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs_float - f_exp)**2 / f_exp
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\scipy\stats\_stats_py.py:8022: Runt

In [70]:
dataset_drift = next(
    m for m in res["metrics"]
    if m["metric"] == "DatasetDriftMetric")

drift_res = dataset_drift["result"]

nb_cols = drift_res["number_of_columns"]
nb_drift = drift_res["number_of_drifted_columns"]
taux = nb_drift / nb_cols

print("Colonnes analysées :", nb_cols)
print("Colonnes en dérive :", nb_drift)
print(f"Taux de dérive : {taux:.1%}")


Colonnes analysées : 275
Colonnes en dérive : 18
Taux de dérive : 6.5%


In [71]:
# Colonnes en dérive
df_drift[df_drift["drift_detected"] == True]


,colonne,drift_detected
6,num__PREV_PRODUCT_COMBINATION_POS others witho...,True
57,num__BUREAU_STATUS_4,True
60,num__PREV_NAME_CASH_LOAN_PURPOSE_Buying a holi...,True
66,num__PREV_NAME_GOODS_CATEGORY_Vehicles,True
84,num__PREV_NAME_CASH_LOAN_PURPOSE_Medicine,True
86,num__PREV_NAME_CASH_LOAN_PURPOSE_Wedding / gif...,True
89,num__DAYS_REGISTRATION,True
94,num__PREV_NAME_GOODS_CATEGORY_Jewelry,True
123,num__PREV_CODE_REJECT_REASON_NUNIQUE,True
133,num__PREV_CODE_REJECT_REASON_SCOFR,True


## **Optimisation des performances du modèle**

### Apperçu global

In [72]:
df_production_raw.head()

,num__SK_ID_CURR,num__BUREAU_SK_ID_BUREAU_max,num__POS_SK_DPD_min,num__CC_SK_DPD_DEF_max,num__PREV_NAME_TYPE_SUITE_Family,num__PREV_DAYS_FIRST_DRAWING_sum,num__PREV_NAME_GOODS_CATEGORY_Other,num__DAYS_LAST_PHONE_CHANGE,num__CC_CNT_DRAWINGS_OTHER_CURRENT_mean,num__PREV_DAYS_FIRST_DRAWING_max,...,ram_percent,system_load,num_threads,method,status,error_message,latency_ms,request_size_bytes,response_size_bytes,batch_size_y
0,-0.302429,0.326039,0.249338,NaN,-0.535936,-0.300165,0.23674,-0.987792,NaN,0.246419,...,78.3,0.0,21,POST,success,None,383.741379,12798.0,0.0,NaN
1,-0.147132,0.553947,0.249338,NaN,-0.535936,0.742804,0.23674,0.895050,NaN,0.246419,...,78.3,0.0,21,POST,success,None,102.715015,12775.0,0.0,NaN
2,-0.429638,0.448635,0.249338,NaN,0.494736,-0.821649,0.23674,0.962033,NaN,0.246419,...,78.4,0.0,21,POST,success,None,67.815304,12875.0,0.0,NaN
3,0.246223,0.491943,0.249338,0.691535,2.556081,4.392488,0.23674,-2.192274,1.997573,0.246419,...,78.4,0.0,21,POST,success,None,89.750528,12927.0,0.0,NaN
4,-1.169600,0.699197,0.249338,NaN,-0.535936,-0.821649,0.23674,0.744033,NaN,0.246419,...,78.3,0.0,21,POST,success,None,68.776608,12836.0,0.0,NaN


In [73]:
# Statistiques inférence
df_production_raw['inference_ms_x']

0      48.473900
1      53.282800
2      22.520600
3      39.389800
4      23.058600
         ...    
208    48.540900
209    45.094800
210    45.515200
211    46.950299
212    52.126600
Name: inference_ms_x, Length: 213, dtype: float64

In [74]:
df_production_raw['inference_ms_x'].describe()

count    213.000000
mean      36.709429
std       15.020234
min       21.098300
25%       24.290100
50%       30.452900
75%       46.725800
max      113.723900
Name: inference_ms_x, dtype: float64

In [75]:
# Statistiques latence
df_production_raw["latency_ms"]

0      383.741379
1      102.715015
2       67.815304
3       89.750528
4       68.776608
          ...    
208    307.635784
209    315.336943
210    302.381754
211    309.991360
212    237.794876
Name: latency_ms, Length: 213, dtype: float64

In [76]:
df_production_raw["latency_ms"].describe()

count     213.000000
mean      104.265481
std        90.755837
min        58.841944
25%        70.789099
50%        78.964949
75%        96.737385
max      1024.606943
Name: latency_ms, dtype: float64

In [77]:
# Statistiques score
df_production_raw["score_x"]

0      0.074383
1      0.670045
2      0.092590
3      0.151949
4      0.043955
         ...   
208         NaN
209         NaN
210         NaN
211         NaN
212    0.162290
Name: score_x, Length: 213, dtype: float64

In [78]:
df_production_raw["score_x"].describe()

count    201.000000
mean       0.213696
std        0.161163
min        0.021356
25%        0.084765
50%        0.190217
75%        0.285706
max        0.793830
Name: score_x, dtype: float64

In [79]:
df_production_raw["decision_x"].value_counts()

decision_x
ACCORDÉ    184
REFUSÉ      17
Name: count, dtype: int64

### Statistiques globales sur les performances API

Les métriques de production montrent que la latence moyenne de l’API est de *104 ms*, avec un p95 à *300 ms* et quelques pics à 1 seconde.

Le CPU reste faible *(24 %)*, mais la RAM est élevée *(74 %)*, ce qui confirme que l’API n’est pas limitée par le modèle mais par le ***preprocessing***.

Le temps d’inférence mesuré **(36 ms en moyenne, 65 ms au p95)** correspond essentiellement au preprocessing sklearn, le modèle lui-même étant très rapide **(< 3 ms)**.

Ces résultats confirment que l’optimisation doit se concentrer sur le preprocessing plutôt que sur le modèle. 

In [80]:
lat_mean = df_production_raw["latency_ms"].mean() # latence moyenne
lat_p95 = df_production_raw["latency_ms"].quantile(0.95) # latence au 95ème percentile
lat_max = df_production_raw["latency_ms"].max() # latence maximale observée

cpu_mean = df_production_raw["cpu_percent"].mean()
ram_mean = df_production_raw["ram_percent"].mean()

infer_mean = df_production_raw["inference_ms_x"].mean() # temps moyen passé dans predict_proba()
infer_p95 = df_production_raw["inference_ms_x"].quantile(0.95) # temps au 95ème percentile

print(f"latency_mean_p95_max: {lat_mean, lat_p95, lat_max}")
print(f"cpu_ram: {cpu_mean, ram_mean}")
print(f"infer: {infer_p95, infer_mean}")

latency_mean_p95_max: (104.26548053401177, 300.34413337707514, 1024.6069431304932)
cpu_ram: (24.00985915492958, 74.26103286384976)
infer: (64.89101992920038, 36.70942912341703)


### Recherche des goulots d'étranglement - **cProfile**

In [81]:
def benchmark():
    for _ in range(100):
        pipe.predict_proba(df_reference_raw)

cProfile.run(
    "benchmark()",
    "profiling.prof")

stats = pstats.Stats("profiling.prof")
stats.sort_stats("cumtime")
stats.print_stats(18)

Sun Jun 14 01:44:09 2026    profiling.prof

         4105904 function calls (4085504 primitive calls) in 9.293 seconds

   Ordered by: cumulative time
   List reduced from 591 to 18 due to restriction <18>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    9.346    9.346 {built-in method builtins.exec}
        1    0.000    0.000    9.345    9.345 <string>:1(<module>)
        1    0.029    0.029    9.345    9.345 C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6608\1097060041.py:1(benchmark)
      100    0.003    0.000    9.314    0.093 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\pipeline.py:676(predict_proba)
  500/100    0.035    0.000    8.744    0.087 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\_set_output.py:293(wrapped)
      100    0.047    0.000    8.708    0.087 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\compose\_column_transformer.py:946(transform)
      100    0.

Résultat profiling sur l'ensemble du dataset.
Sat Jun 13 23:37:16 2026    profiling.prof

         150875804 function calls (150855404 primitive calls) in 170.527 seconds

   Ordered by: cumulative time
   List reduced from 593 to 18 due to restriction <18>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000  170.568  170.568 {built-in method builtins.exec}
        1    0.000    0.000  170.568  170.568 <string>:1(<module>)
        1    2.378    2.378  170.568  170.568 C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6608\4137855541.py:4(benchmark)
      100    0.002    0.000  168.186    1.682 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\pipeline.py:676(predict_proba)
  500/100    1.023    0.002  145.599    1.456 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\_set_output.py:293(wrapped)
      100    1.367    0.014  144.604    1.446 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\compose\_column_transformer.py:946(transform)
      100    0.009    0.000  136.908    1.369 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\compose\_column_transformer.py:751(_call_func_on_transformers)
      100    0.001    0.000  129.304    1.293 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\parallel.py:44(__call__)
      100    0.002    0.000  129.303    1.293 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\joblib\parallel.py:1969(__call__)
      600    0.005    0.000  129.296    0.215 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\joblib\parallel.py:1888(_get_sequential_output)
      400    0.004    0.000  129.288    0.323 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\parallel.py:115(__call__)
      400    0.003    0.000  129.261    0.323 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\pipeline.py:1261(_transform_one)
      200    0.723    0.004  114.522    0.573 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\preprocessing\_encoders.py:185(_transform)
      100    0.360    0.004   99.135    0.991 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\preprocessing\_encoders.py:984(transform)
     1600    2.995    0.002   92.976    0.058 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\_encode.py:236(_check_unknown)
     3200    0.187    0.000   66.984    0.021 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\_encode.py:108(_extract_missing)
     3200    7.114    0.002   66.791    0.021 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\_encode.py:124(<setcomp>)
 11946300   15.667    0.000   59.734    0.000 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\__init__.py:1151(is_scalar_nan)

Le profiling sur l'ensemble du dataset (plus de 40000 lignes) montre que la latence du pipeline est dominée par le **preprocessing (85 % du temps)**, en particulier par *OneHotEncoder et ses vérifications internes (_check_unknown, is_scalar_nan)*:
- `ColumnTransformer.transform` <> 144.6 s cumulés
- `OneHotEncoder._transform` <> 114.5 s ; représente **67%** du temps total (170.6 ms) - très coûteux en unitaire
- `OneHotEncoder.transform` <> 99.1 s
- `_check_unknown` <> 92.9 s
- `is_scalar_nan` <> 59.7 s

Le profiling sur un échantilon (500 lignes) pour des besoins GiHub montre:
- un coût total réduit (9 s au lieu de 170 s)
- des fonctions qui disparaissent car moins de lignes
- un comportement plus “léger”
- un pipeline qui reste coûteux mais moins extrême
Néanmoins, la structure du coût reste la même (preprocessing dominant), mais :
- les appels à `_check_unknown` et `is_scalar_nan` chutent fortement,
- les fonctions liées aux valeurs rares disparaissent du top 20,
- le coût absolu est beaucoup plus faible.

Les deux profilings sont complémentaires :
- le profiling complet montre le comportement réel du pipeline en production,
- le profiling réduit montre le comportement dans un environnement GitHub/CI.

Ils démontrent ensemble que :
- **le modèle est très rapide**,  
- **le preprocessing est le goulot d’étranglement**,  
- **le coût dépend fortement de la taille du dataset**,  
- **le batch est indispensable pour réduire la latence**.

### Préprocessing vs modèle
Le modèle lui‑même est très rapide *(< 3 ms)*, mais le preprocessing coûte *23 ms par ligne*, ce qui explique les latences élevées en appels unitaires.

In [82]:
try:
    df_example = df_reference_raw
except:
    df_example = None

X = df_example.iloc[[0]]
t0 = time.perf_counter()

X_transformed = pipe[:-1].transform(X)
t1 = time.perf_counter()

score = pipe[-1].predict_proba(X_transformed)
t2 = time.perf_counter()

print("preprocessing :", (t1 - t0)*1000)
print("model :", (t2 - t1)*1000)
print("total :", (t2 - t0)*1000)

preprocessing : 19.87539976835251
model : 33.621800132095814
total : 53.49719990044832


### Optimisation du preprocessing: unitaire vs batch

Les résultats ci-dessous confirment le profiling:
- 1 ligne > **23 ms** de preprocessing
- 100 lignes en unitaire > **2266 ms**
- 100 lignes en batch > **31 ms**

Vectorisé, le preprocessing ne s’exécute qu’une seule fois en batch, mais 100 fois en unitaire <> Le batch est sensiblement **70 fois plus rapide que l’unitaire**.

In [83]:
start = time.perf_counter()

for i in range(100):
    pipe.predict_proba(df_example.iloc[[i]])

end = time.perf_counter()
print("100 appels unitaires :", (end - start) * 1000, "ms")

100 appels unitaires : 2384.8198000341654 ms


In [84]:
X_batch = df_example.iloc[:100]

start = time.perf_counter()
pipe.predict_proba(X_batch)
end = time.perf_counter()

print("1 batch de 100 lignes :", (end - start) * 1000, "ms")

1 batch de 100 lignes : 30.14650009572506 ms


Le batch permet de vectoriser les transformations et réduit le temps total d’un facteur ×70.
Ces résultats démontrent que l’optimisation doit se concentrer sur le preprocessing plutôt que sur le modèle.

# **Conclusion**

**1. Analyse du data drift**

Après harmonisation complète des colonnes entre les données de référence et les données de production (conversion des types, gestion des valeurs manquantes, suppression des colonnes constantes, alignement des colonnes communes), nous avons pu analyser le drift sur **278 colonnes**.

Les résultats montrent :
- **Colonnes testées : 278**
- **Colonnes avec drift détecté : 79**
- **Taux de dérive : 28.4 %**

Ce taux est significatif : près d’un tiers des variables présentent une distribution différente entre la référence et la production.  
Les colonnes les plus touchées concernent principalement :
- les variables dérivées de comportements (durées, montants, ratios),
- certaines variables catégorielles encodées,
- et des variables liées à l’historique de crédit.

Un tel niveau de drift peut potentiellement impacter la stabilité du modèle, et justifie la mise en place d’un monitoring continu ainsi que d’un mécanisme d’alerte.

**2. Analyse de la performance du pipeline**

Les mesures de latence montrent que :

- **le preprocessing est de loin la partie la plus coûteuse** du pipeline (23 ms par ligne, contre 2.7 ms pour le modèle lui-même),
- **100 appels unitaires** coûtent **2266 ms**, car le preprocessing est répété 100 fois,
- **un batch de 100 lignes** ne coûte que **31 ms**, grâce à la vectorisation des transformations.

Ces résultats sont confirmés par le profiling, qui montre que :

- **85 % du temps total** est passé dans `ColumnTransformer.transform`,
- dont **67 % dans OneHotEncoder** et ses vérifications internes (`_check_unknown`, `is_scalar_nan`).

Le modèle, lui, est très rapide (< 3 ms).